# Download and Preprocess dbNSFP Variant Files

Convert the variant .gz files to parquet files for faster processing and merge with our ClinVar annotations

In [1]:
import os
import polars as pl
import pandas as pd

## 1. dbNSFP

### 1.1 Get the prediction scores from 

Using ../1_inputs/download_data.sh to download data from https://drive.google.com/file/d/1HNoZUswadC1ODavU7AUvm0DZ13jKVFKW/view

### 1.2 Convert the chr variant files to parquet

__For easier accessing and processing__

In [ ]:
# def convert_to_parquet(chr_num):
#     file_path = f"../1_inputs/raw_inputs/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.gz"
#     df = pl.read_csv(file_path, 
#                      separator='\t',
#                      n_threads=16,
#                      infer_schema_length=1000000,
#                      ignore_errors=True,
#                      truncate_ragged_lines=True,
#                      low_memory=False)
#     df = df.with_columns(
#         pl.col("aaref").replace(aa_single_to_three).alias("aaref_3"),
#         pl.col("aaalt").replace(aa_single_to_three).alias("aaalt_3"),
#     )
#     parquet_path = f"../1_inputs/raw_inputs/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.parquet"
#     df.write_parquet(parquet_path)
#     return parquet_path

# chr_num_list = [chr_file.split('.')[-2].split('chr')[-1] for chr_file in os.listdir('../1_inputs/raw_inputs/dbNSFP5.0a') if "chr" in chr_file and chr_file.endswith(".gz")]
# # Use ThreadPoolExecutor for parallel processing
# with concurrent.futures.ThreadPoolExecutor() as executor:
#     futures = [executor.submit(convert_to_parquet, chr_num) for chr_num in chr_num_list]
#     # Collect the results
#     parquet_files = [future.result() for future in concurrent.futures.as_completed(futures)]

# print("Conversion to Parquet completed.")

### 1.3 Merge with ClinVar

Merging the most up-to-date clinvar annotations (ClinVar variants released on 2024-03-01 and the data is accessed on March 13, 2025) to dbNSFP dataset for consistency.

#### Get the original ClinVar Annotations

Please refer to ```../../1_allele_collection/1_inputs/download_clinvar_gnomad.ipynb``` for more details about ClinVar annotations.

In [2]:
processed_clinvar_parq = "../../1_allele_collection/3_outputs/processed_data/clinvar/clinvar_processed.parquet"
clinvar = pl.scan_parquet(processed_clinvar_parq)

In [4]:
clinvar.select(pl.col("Type")).unique().collect().to_numpy()

array([['Duplication'],
       ['copy number gain'],
       ['Microsatellite'],
       ['Translocation'],
       ['Insertion'],
       ['Deletion'],
       ['Indel'],
       ['Inversion'],
       ['Complex'],
       ['single nucleotide variant'],
       ['copy number loss'],
       ['Variation']], dtype=object)

In [5]:
clinvar.collect().shape

(3390885, 47)

In [3]:
clinvar_snv = clinvar.filter(pl.col("Type")=='single nucleotide variant') ##.select(pl.col("Type")).unique().collect().to_numpy()
clinvar_snv.collect().shape

(3105660, 47)

In [4]:
clinvar_snv.collect()

#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,SPDI,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
i64,str,str,i64,str,str,cat,i64,str,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
15043,"""single nucleotide variant""","""NM_014630.3(ZNF592):c.3136G>A …",9640,"""ZNF592""","""HGNC:28986""","""Uncertain significance""",0,"""Jun 29, 2015""",150829393,"""-""","""RCV000000014""","""MONDO:MONDO:0033005,MedGen:C45…","""Galloway-Mowat syndrome 1""","""germline""","""germline""","""GRCh38""","""NC_000015.10""","""15""",84799209,84799209,"""15q25.3""","""no assertion criteria provided""",1,"""-""","""N""","""ClinGen:CA210674,UniProtKB:Q92…",1,4,84799209,"""G""","""A""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000020157""","""-""","""-""","""3136G>A ""","""Gly1046Arg""","""NC_000015.10:84799209:G:A""","""NM_014630.3""",null,"""4_VUS"""
15044,"""single nucleotide variant""","""NM_017547.4(FOXRED1):c.694C>T …",55572,"""FOXRED1""","""HGNC:26927""","""Pathogenic""",1,"""Mar 12, 2024""",267606829,"""-""","""RCV000000015|RCV000578659|RCV0…","""MONDO:MONDO:0032624,MedGen:C47…","""Mitochondrial complex 1 defici…","""germline""","""germline""","""GRCh38""","""NC_000011.10""","""11""",126275389,126275389,"""11q24.2""","""criteria provided, multiple su…",6,"""-""","""N""","""ClinGen:CA113792,OMIM:613622.0…",3,5,126275389,"""C""","""T""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000680696|SCV001363290|SCV0…","""-""","""-""","""694C>T ""","""Gln232Ter""","""NC_000011.10:126275389:C:T""","""NM_017547.4""",2,"""1_Pathogenic"""
15045,"""single nucleotide variant""","""NM_017547.4(FOXRED1):c.1289A>G…",55572,"""FOXRED1""","""HGNC:26927""","""Likely pathogenic""",1,"""Jun 06, 2024""",267606830,"""-""","""RCV000000016""","""MONDO:MONDO:0032624,MedGen:C47…","""Mitochondrial complex 1 defici…","""germline""","""germline""","""GRCh38""","""NC_000011.10""","""11""",126277517,126277517,"""11q24.2""","""criteria provided, single subm…",2,"""-""","""N""","""ClinGen:CA113794,UniProtKB:Q96…",3,6,126277517,"""A""","""G""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV005680614""","""-""","""-""","""1289A>G ""","""Asn430Ser""","""NC_000011.10:126277517:A:G""","""NM_017547.4""",1,"""1_Pathogenic"""
15046,"""single nucleotide variant""","""NM_025152.3(NUBPL):c.166G>A (p…",80224,"""NUBPL""","""HGNC:20278""","""Conflicting classifications of…",1,"""Nov 12, 2024""",200401432,"""-""","""RCV000196589|RCV000622708|RCV0…","""MedGen:C3661900|MeSH:D030342,M…","""not provided|Inborn genetic di…","""germline;paternal""","""germline""","""GRCh38""","""NC_000014.9""","""14""",31562125,31562125,"""14q12""","""criteria provided, conflicting…",5,"""-""","""N""","""OMIM:613621.0001,ClinGen:CA321…",2,214885,31562125,"""G""","""A""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000251971|SCV000742093|SCV0…","""-""","""-""","""166G>A ""","""Gly56Arg""","""NC_000014.9:31562125:G:A""","""NM_025152.3""",1,"""3_Conflicting"""
15048,"""single nucleotide variant""","""NM_000410.4(HFE):c.845G>A (p.C…",3077,"""HFE""","""HGNC:4886""","""Pathogenic/Pathogenic, low pen…",1,"""Mar 04, 2025""",1800562,"""-""","""RCV000000019|RCV000210820|RCV0…","""MONDO:MONDO:0021001,MedGen:C34…","""Hemochromatosis type 1|Heredit…","""biparent

In [22]:
clinvar_snv.filter(pl.col("VariationID")==2333487).collect()

#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,SPDI,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
i64,str,str,i64,str,str,cat,i64,str,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
2324965,"""single nucleotide variant""","""NM_199261.4(TPTE):c.37G>C (p.V…",7179,"""TPTE""","""HGNC:12023""","""Uncertain significance""",0,"""Aug 02, 2021""",-1,"""-""","""RCV004175219""","""MedGen:CN169374""","""not specified""","""germline""","""germline""","""GRCh38""","""NC_000021.9""","""21""",10541137,10541137,"""21p11.2""","""criteria provided, single subm…",1,"""-""","""N""","""ClinGen:CA9978559""",2,2333487,10541137,"""G""","""C""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV003666233""","""-""","""-""","""37G>C ""","""Val13Leu""","""NC_000021.9:10541137:G:C""","""NM_199261.4""",1,"""4_VUS"""


In [8]:
clinvar_snv_collect = clinvar_snv.collect()
clinvar_snv_collect["VariationID"].unique()

VariationID
i64
4
5
6
9
10
…
3767244
3767247
3767248


#### Merge dbNSFP and ClinVar

dbNSFP and ClinVar datasets were merged together using ```VariantionID``` (ClinVar) and ```clinvar_id``` (dbNSFP), which are the unique identifier for each variant.

In [ ]:
%%time

# with pl.StringCache():
#     chr_num_list = [chr_file.split('.')[-2].split('chr')[-1] for chr_file in os.listdir('./raw_inputs/dbNSFP5.0a/') if "chr" in chr_file and chr_file.endswith(".gz")]
#     clin_var_dfs = []
#     for chr_num in chr_num_list:
#         df_chr_par = pl.scan_parquet(f"./raw_inputs/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.parquet")
#         df_chr_par_clinvar = df_chr_par.filter(pl.col("clinvar_id")!='.')
#         clinvar_per_chr = clinvar_snv.filter(pl.col("Chromosome")==chr_num)#.collect()
#         clinvar_per_chr = clinvar_per_chr.with_columns(pl.col("VariationID").cast(pl.String).alias("clinvar_id"))
#         df_chr_par_clinvar = df_chr_par_clinvar.join(clinvar_per_chr, on="clinvar_id", how="left")
#         df_chr_par_clinvar = df_chr_par_clinvar.collect()
#         clin_var_dfs.append(df_chr_par_clinvar)

#     df_clinvar = pl.concat(clin_var_dfs, how="vertical_relaxed") ## 
#     parquet_path = f"../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot.parquet"
#     df_clinvar.write_parquet(parquet_path)

CPU times: user 45min 17s, sys: 36min 5s, total: 1h 21min 23s
Wall time: 1min 22s


## 2. Deduplicate the variants

### 2.1 Explode variants

Many variants exist on multiple isoforms and are all aggregated together, and we need to split them apart for easier analyses.

In [12]:
df_clinvar = pl.scan_parquet("../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot.parquet")
# For testing purposes, limit to a small subset:
df_clin_subset = df_clinvar.limit(10)
# Collect the subset into an eager DataFrame for inspection
df_clin_subset = df_clin_subset.collect()

# Automatically detect columns with semicolon-separated patterns.
# We only check columns of Utf8 (string) type.
pattern_cols = [
    col for col in df_clinvar.collect_schema() 
    if df_clin_subset[col].dtype == pl.Utf8 and df_clin_subset[col].str.contains(";").any()
]
print("Columns containing ';':", pattern_cols)

df_clinvar_exploded = df_clinvar.with_columns(
    [pl.col(col).str.split(";").alias(col) for col in pattern_cols]
).explode(pattern_cols)

# df_clinvar_exploded.collect()

Columns containing ';': ['aapos', 'genename', 'Ensembl_geneid', 'Ensembl_transcriptid', 'Ensembl_proteinid', 'Uniprot_acc', 'Uniprot_entry', 'HGVSc_snpEff', 'HGVSp_snpEff', 'HGVSc_VEP', 'HGVSp_VEP', 'APPRIS', 'GENCODE_basic', 'TSL', 'VEP_canonical', 'MANE', 'cds_strand', 'refcodon', 'codonpos', 'codon_degeneracy', 'SIFT_score', 'SIFT_pred', 'SIFT4G_score', 'SIFT4G_pred', 'Polyphen2_HDIV_score', 'Polyphen2_HDIV_pred', 'Polyphen2_HVAR_score', 'Polyphen2_HVAR_pred', 'MutationTaster_score', 'MutationTaster_pred', 'MutationTaster_model', 'MutationTaster_trees_benign', 'MutationTaster_trees_deleterious', 'MutationAssessor_score', 'MutationAssessor_pred', 'PROVEAN_score', 'PROVEAN_pred', 'VEST4_score', 'MetaRNN_score', 'MetaRNN_pred', 'REVEL_score', 'MutPred_score', 'MutPred_protID', 'MutPred_AAchange', 'MutPred_Top5features', 'MVP_score', 'gMVP_score', 'MPC_score', 'DEOGEN2_score', 'DEOGEN2_pred', 'LIST-S2_score', 'LIST-S2_pred', 'VARITY_R_score', 'VARITY_ER_score', 'VARITY_R_LOO_score', 'VA

#chr,pos(1-based),ref,alt,aaref,aaalt,rs_dbSNP,hg19_chr,hg19_pos(1-based),hg18_chr,hg18_pos(1-based),aapos,genename,Ensembl_geneid,Ensembl_transcriptid,Ensembl_proteinid,Uniprot_acc,Uniprot_entry,HGVSc_snpEff,HGVSp_snpEff,HGVSc_VEP,HGVSp_VEP,APPRIS,GENCODE_basic,TSL,VEP_canonical,MANE,cds_strand,refcodon,codonpos,codon_degeneracy,Ancestral_allele,AltaiNeandertal,Denisova,VindijiaNeandertal,ChagyrskayaNeandertal,SIFT_score,…,nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,SPDI,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
"""21""",10541137,"""G""","""C""","""V""","""L""","""rs150733676""",""".""",""".""",""".""",""".""","""13""","""TPTE""","""ENSG00000274391""","""ENST00000622113""","""ENSP00000482040""","""P56180-2""","""TPTE_HUMAN""","""c.37G>C""","""p.Val13Leu""","""c.37G>C""","""p.Val13Leu""","""alternative2""","""Y""","""1""",""".""",""".""","""+""","""GTC""","""1""","""0""","""g""","""./.""","""./.""","""./.""","""./.""",""".""",…,"""-""","""RCV004175219""","""MedGen:CN169374""","""not specified""","""germline""","""germline""","""GRCh38""","""NC_000021.9""","""21""",10541137,10541137,"""21p11.2""","""criteria provided, single subm…",1,"""-""","""N""","""ClinGen:CA9978559""",2,2333487,10541137,"""G""","""C""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV003666233""","""-""","""-""","""37G>C ""","""Val13Leu""","""NC_000021.9:10541137:G:C""","""NM_199261.4""",1,"""4_VUS"""
"""21""",10541137,"""G""","""C""","""V""","""L""","""rs150733676""",""".""",""".""",""".""",""".""","""13""","""TPTE""","""ENSG00000274391""","""ENST00000612957""","""ENSP00000484064""","""A0A087X1C4""","""A0A087X1C4_HUMAN""","""c.37G>C""","""p.Val13Leu""","""c.37G>C""","""p.Val13Leu""",""".""",""".""","""5""",""".""",""".""","""+""","""GTC""","""1""","""0""","""g""","""./.""","""./.""","""./.""","""./.""",""".""",…,"""-""","""RCV004175219""","""MedGen:CN169374""","""not specified""","""germline""","""germline""","""GRCh38""","""NC_000021.9""","""21""",10541137,10541137,"""21p11.2""","""criteria provided, single subm…",1,"""-""","""N""","""ClinGen:CA9978559""",2,2333487,10541137,"""G""","""C""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV003666233""","""-""","""-""","""37G>C ""","""Val13Leu""","""NC_000021.9:10541137:G:C""","""NM_199261.4""",1,"""4_VUS"""
"""21""",10541137,"""G""","""C""","""V""","""L""","""rs150733676""",""".""",""".""",""".""",""".""","""13""","""TPTE""","""ENSG00000274391""","""ENST00000427445""","""ENSP00000482488""","""P56180-3""","""TPTE_HUMAN""","""c.37G>C""","""p.Val13Leu""","""c.37G>C""","""p.Val13Leu""","""alternative2""","""Y""","""1""",""".""",""".""","""+""","""GTC""","""1""","""0""","""g""","""./.""","""./.""","""./.""","""./.""",""".""",…,"""-""","""RCV004175219""","""MedGen:CN169374""","""not specified""","""germline""","""germline""","""GRCh38""","""NC_000021.9""","""21""",10541137,10541137,"""21p11.2""","""criteria provided, single subm…",1,"""-""","""N""","""ClinGen:CA9978559""",2,2333487,10541137,"""G""","""C""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV003666233""","""-""","""-""","""37G>C ""","""Val13Leu""","""NC_000021.9:10541137:G:C""","""NM_199261.4""",1,"""4_VUS""

### 2.2 De-duplicate them 

__By prioritizing the uniprot_id with a .pdb structure inferred by AF2__

In [ ]:
def deduplicate_by_uniprot_lazy(ldf, preferred_uniprot_acc):
    # Create window specs for our operations
    window_spec = pl.col("clinvar_id")
    
    with pl.StringCache():
        # Add useful columns for filtering in a single pass:
        # 1. Count occurrences of each clinvar_id
        # 2. Check if Uniprot_acc is preferred
        # 3. Check if Uniprot_acc has no dash
        # 4. Create row number within each group to use for selection
        enhanced_ldf = ldf.with_columns([
            pl.col("clinvar_id").count().over(window_spec).alias("group_count"),
            pl.col("Uniprot_acc").is_in(preferred_uniprot_acc).alias("is_preferred"),
            (~pl.col("Uniprot_acc").str.contains("-")).alias("no_dash"),
            pl.col("Uniprot_acc").rank("dense").over(window_spec).alias("row_rank")
        ])
        
        # For non-duplicated entries - keep all
        non_duplicated = enhanced_ldf.filter(pl.col("group_count") == 1)
        
        # For duplicated entries, we'll create a priority column and then use that for selection
        duplicated = enhanced_ldf.filter(pl.col("group_count") > 1)

        # Create a priority column (1=preferred, 2=no dash, 3=other)
        prioritized = duplicated.with_columns([
            pl.when(pl.col("is_preferred"))
            .then(pl.lit(1))
            .when(pl.col("no_dash"))
            .then(pl.lit(2))
            .otherwise(pl.lit(3))
            .alias("priority")
        ])

        # Sort by priority and take the first record for each ID
        deduped = (
            prioritized
            .sort(["clinvar_id", "priority", "row_rank"])
            .group_by("clinvar_id")
            .agg(
                pl.all().exclude(["group_count", "is_preferred", "no_dash", "row_rank", "priority"]).first()
            )
        )

    deduped_df = pl.concat([non_duplicated.drop(["group_count", "is_preferred", "no_dash", "row_rank"]).select(pl.col(deduped.collect_schema().names())),
    deduped], how="vertical_relaxed")
    
    # Combine all results
    return deduped_df

uniprot_ids = [f.split('-')[1] for f in os.listdir("../1_inputs/raw_inputs/alphafold/UP000005640_9606_HUMAN_v4") if f.endswith("pdb.gz")]
## Let's prioritize the variants with PDB structures
df_clinvar_dedup_struct = deduplicate_by_uniprot_lazy(df_clinvar_exploded, uniprot_ids)

In [ ]:
# with pl.StringCache():
#     df_clinvar_dedup_struct.collect().write_parquet(f"../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot_pdb_variants.parquet")

In [27]:
df_clinvar_dedup_struct = pl.scan_parquet("../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot_pdb_variants.parquet")
df_clinvar_dedup_struct.collect()

clinvar_id,#chr,pos(1-based),ref,alt,aaref,aaalt,rs_dbSNP,hg19_chr,hg19_pos(1-based),hg18_chr,hg18_pos(1-based),aapos,genename,Ensembl_geneid,Ensembl_transcriptid,Ensembl_proteinid,Uniprot_acc,Uniprot_entry,HGVSc_snpEff,HGVSp_snpEff,HGVSc_VEP,HGVSp_VEP,APPRIS,GENCODE_basic,TSL,VEP_canonical,MANE,cds_strand,refcodon,codonpos,codon_degeneracy,Ancestral_allele,AltaiNeandertal,Denisova,VindijiaNeandertal,ChagyrskayaNeandertal,…,nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,SPDI,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
"""2361281""","""21""",13628442,"""G""","""T""","""S""","""I""","""rs905107791""","""21""","""15000763""","""21""","""13922634""","""396""","""POTED""","""ENSG00000166351""","""ENST00000299443""","""ENSP00000299443""","""Q86YR6""","""POTED_HUMAN""","""c.1187G>T""","""p.Ser396Ile""","""c.1187G>T""","""p.Ser396Ile""","""principal1""","""Y""","""1""","""YES""","""Select""","""+""","""AGC""","""2""","""0""",""".""","""./.""","""./.""","""./.""","""./.""",…,"""-""","""RCV004201236""","""MedGen:CN169374""","""not specified""","""germline""","""germline""","""GRCh38""","""NC_000021.9""","""21""",13628442,13628442,"""21q11.2""","""criteria provided, single subm…",1,"""-""","""N""","""ClinGen:CA317768197""",2,2361281,13628442,"""G""","""T""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV003702401""","""-""","""-""","""1187G>T ""","""Ser396Ile""","""NC_000021.9:13628442:G:T""","""NM_174981.6""",1,"""4_VUS"""
"""2599176""","""21""",13630941,"""G""","""T""","""V""","""F""","""rs937658823""","""21""","""15003262""","""21""","""13925133""","""415""","""POTED""","""ENSG00000166351""","""ENST00000299443""","""ENSP00000299443""","""Q86YR6""","""POTED_HUMAN""","""c.1243G>T""","""p.Val415Phe""","""c.1243G>T""","""p.Val415Phe""","""principal1""","""Y""","""1""","""YES""","""Select""","""+""","""GTT""","""1""","""0""",""".""","""./.""","""./.""","""./.""","""./.""",…,"""-""","""RCV004345528""","""MedGen:CN169374""","""not specified""","""germline""","""germline""","""GRCh38""","""NC_000021.9""","""21""",13630941,13630941,"""21q11.2""","""criteria provided, single subm…",1,"""-""","""N""","""ClinGen:CA317768279""",2,2599176,13630941,"""G""","""T""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV004065319""","""-""","""-""","""1243G>T ""","""Val415Phe""","""NC_000021.9:13630941:G:T""","""NM_174981.6""",1,"""4_VUS"""
"""2486768""","""21""",13630977,"""G""","""A""","""V""","""M""",""".""","""21""","""15003298""","""21""","""13925169""","""427""","""POTED""","""ENSG00000166351""","""ENST00000299443""","""ENSP00000299443""","""Q86YR6""","""POTED_HUMAN""","""c.1279G>A""","""p.Val427Met""","""c.1279G>A""","""p.Val427Met""","""principal1""","""Y""","""1""","""YES""","""Select""","""+""","""GTG""","""1""","""0""",""".""","""./.""","""./.""","""./.""","""./.""",…,"""-""","""RCV004275341""","""MedGen:CN169374""","""not specified""","""germline""","""germline""","""GRCh38""","""NC_000021.9""","""21""",13630977,13630977,"""21q11.2""","""criteria provided, single subm…",1,"""-""","""N""","""ClinGen:CA409859409""",2,2486768,13630977,"""G""","""A""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV003905383""

#### De-duplicate them by simply keeping the first occurrence of gene_id and AA change

Not a very good idea. Deprecated.

In [1]:
# df_clinvar_dedup_var = df_clinvar_exploded.collect().unique(subset=["Ensembl_geneid", "HGVSp_snpEff"], keep="first")
# # df_clinvar_dedup_var
# df_clinvar_dedup_var.write_parquet(f"../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot_dedup_variants.parquet")